In [1]:
import json
import re

In [2]:
def fix_latex(input_file, output_file):
    """
    Fix Unicode escape sequences and convert special characters to LaTeX in JSONL file.
    Wraps LaTeX math in single $...$ delimiters.
    """
    
    # Mapping for Unicode to LaTeX conversion
    unicode_to_latex = {
        # Greek letters (lowercase)
        'α': r'\alpha', 'β': r'\beta', 'γ': r'\gamma', 'δ': r'\delta',
        'ε': r'\epsilon', 'ζ': r'\zeta', 'η': r'\eta', 'θ': r'\theta',
        'ι': r'\iota', 'κ': r'\kappa', 'λ': r'\lambda', 'μ': r'\mu',
        'ν': r'\nu', 'ξ': r'\xi', 'π': r'\pi', 'ρ': r'\rho',
        'σ': r'\sigma', 'τ': r'\tau', 'υ': r'\upsilon', 'φ': r'\phi',
        'χ': r'\chi', 'ψ': r'\psi', 'ω': r'\omega',
        
        # Greek letters (uppercase)
        'Γ': r'\Gamma', 'Δ': r'\Delta', 'Θ': r'\Theta', 'Λ': r'\Lambda',
        'Ξ': r'\Xi', 'Π': r'\Pi', 'Σ': r'\Sigma', 'Φ': r'\Phi',
        'Ψ': r'\Psi', 'Ω': r'\Omega',
        
        # Math symbols
        '×': r'\times', '÷': r'\div', '±': r'\pm', '∓': r'\mp',
        '≈': r'\approx', '≠': r'\neq', '≤': r'\leq', '≥': r'\geq',
        '∞': r'\infty', '∫': r'\int', '∑': r'\sum', '∏': r'\prod',
        '√': r'\sqrt', '∂': r'\partial', '∇': r'\nabla',
        '∈': r'\in', '∉': r'\notin', '⊂': r'\subset', '⊃': r'\supset',
        '∪': r'\cup', '∩': r'\cap', '∧': r'\land', '∨': r'\lor',
        '¬': r'\neg', '→': r'\rightarrow', '←': r'\leftarrow',
        '↔': r'\leftrightarrow', '⇒': r'\Rightarrow', '⇐': r'\Leftarrow',
        '⇔': r'\Leftrightarrow', '∀': r'\forall', '∃': r'\exists',
        '∅': r'\emptyset', '⊥': r'\perp', '∥': r'\parallel',
        '∝': r'\propto', '≡': r'\equiv', '≅': r'\cong', '∼': r'\sim',
        
        # Unicode minus to regular minus
        '−': '-',
        
        # Em dash to regular dash  
        '—': '--',
        
        # Degree symbol
        '°': r'^\circ',
        
        # Superscripts
        '⁰': '^0', '¹': '^1', '²': '^2', '³': '^3', '⁴': '^4',
        '⁵': '^5', '⁶': '^6', '⁷': '^7', '⁸': '^8', '⁹': '^9',
        '⁺': '^+', '⁻': '^-', 'ⁿ': '^n',
        
        # Subscripts
        '₀': '_0', '₁': '_1', '₂': '_2', '₃': '_3', '₄': '_4',
        '₅': '_5', '₆': '_6', '₇': '_7', '₈': '_8', '₉': '_9',
        'ₐ': '_a', 'ₑ': '_e', 'ₕ': '_h', 'ᵢ': '_i', 'ⱼ': '_j',
        'ₖ': '_k', 'ₗ': '_l', 'ₘ': '_m', 'ₙ': '_n', 'ₒ': '_o',
        'ₚ': '_p', 'ᵣ': '_r', 'ₛ': '_s', 'ₜ': '_t', 'ᵤ': '_u',
        'ᵥ': '_v', 'ₓ': '_x',
    }
    
    # LaTeX commands that indicate math mode
    latex_commands = [
        'alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta', 'eta', 'theta',
        'iota', 'kappa', 'lambda', 'mu', 'nu', 'xi', 'pi', 'rho', 'sigma',
        'tau', 'upsilon', 'phi', 'chi', 'psi', 'omega',
        'Gamma', 'Delta', 'Theta', 'Lambda', 'Xi', 'Pi', 'Sigma', 'Phi', 'Psi', 'Omega',
        'times', 'div', 'pm', 'mp', 'approx', 'neq', 'leq', 'geq', 'infty',
        'int', 'sum', 'prod', 'sqrt', 'partial', 'nabla', 'in', 'notin',
        'subset', 'supset', 'cup', 'cap', 'land', 'lor', 'neg',
        'rightarrow', 'leftarrow', 'leftrightarrow',
        'Rightarrow', 'Leftarrow', 'Leftrightarrow',
        'forall', 'exists', 'emptyset', 'perp', 'parallel', 'propto',
        'equiv', 'cong', 'sim', 'circ', 'cdot', 'frac', 'exp', 'log', 'ln',
    ]
    
    def convert_unicode_to_latex(text):
        """Convert Unicode characters to LaTeX commands."""
        for unicode_char, latex_cmd in unicode_to_latex.items():
            text = text.replace(unicode_char, latex_cmd)
        return text
    
    def convert_html_to_latex(text):
        """Convert HTML tags to LaTeX."""
        text = re.sub(r'<sup>(.*?)</sup>', r'^{\1}', text)
        text = re.sub(r'<sub>(.*?)</sub>', r'_{\1}', text)
        return text
    
    def clean_markdown(text):
        """Remove markdown formatting."""
        text = re.sub(r'\*\^{([^}]+)}\*', r'^{\1}', text)
        text = re.sub(r'\*_{([^}]+)}\*', r'_{\1}', text)
        text = re.sub(r'\*([A-Za-z])\*', r'\1', text)
        text = re.sub(r'\*\(([^)]+)\)\*', r'(\1)', text)
        return text
    
    def wrap_latex_in_dollars(text):
        """Wrap LaTeX commands in $...$."""
        
        # Protect existing math regions
        protected = []
        
        def protect(match):
            protected.append(match.group(0))
            return f'\x00MATH{len(protected)-1}\x00'
        
        # Protect existing $$...$$ and $...$
        text = re.sub(r'\$\$[^$]+\$\$', protect, text)
        text = re.sub(r'\$[^$]+\$', protect, text)
        
        # Build pattern - wrap \command with optional trailing ^{} or _{}
        cmds_sorted = sorted(latex_commands, key=len, reverse=True)
        cmd_pattern = '|'.join(re.escape(cmd) for cmd in cmds_sorted)
        
        # Pattern: \command followed by optional spaces and ^{...} or _{...} sequences
        # Also capture optional single letter/digit after command
        pattern = r'\\(' + cmd_pattern + r')((?:\s*[\^_](?:\{[^{}]*\}|[a-zA-Z0-9]))*)'
        
        def wrap(match):
            cmd = match.group(1)
            suffix = match.group(2) if match.group(2) else ''
            return f'$\\{cmd}{suffix}$'
        
        text = re.sub(pattern, wrap, text)
        
        # Restore protected regions (convert $$ to $)
        for idx, region in enumerate(protected):
            if region.startswith('$$') and region.endswith('$$'):
                inner = region[2:-2]
                region = f'${inner}$'
            text = text.replace(f'\x00MATH{idx}\x00', region)
        
        return text
    
    def merge_adjacent_math(text):
        """Merge adjacent $...$ regions separated by math characters."""
        # Merge $a$ $b$ -> $a b$
        # Merge $a$+$b$ -> $a+b$
        # Merge $a$ = $b$ -> $a = b$
        
        # Pattern: $content$ followed by math-like separator followed by $content$
        # Math separators: + - * / = < > ( ) space
        prev = None
        while prev != text:
            prev = text
            # Merge with operators between
            text = re.sub(r'\$([^$]+)\$\s*([+\-*/=<>])\s*\$([^$]+)\$', r'$\1 \2 \3$', text)
            # Merge with just spaces between (if short)
            text = re.sub(r'\$([^$]{1,20})\$\s+\$([^$]{1,20})\$', r'$\1 \2$', text)
            # Merge with parentheses
            text = re.sub(r'\$([^$]+)\$\s*\(\s*\$([^$]+)\$\s*\)', r'$\1(\2)$', text)
        
        return text
    
    def post_process(text):
        """Final cleanup."""
        # Fix e-$\lambda^{t}$ -> $e^{-\lambda t}$ (when ^{} is inside the $)
        text = re.sub(r'e-\$\\([a-zA-Z]+)\^{([^}]*)}\$', r'$e^{-\\\1 \2}$', text)
        
        # Fix e-$\lambda$^{t} -> $e^{-\lambda t}$ (when ^{} is outside the $)  
        text = re.sub(r'e-\$\\([a-zA-Z]+)\$\^{([^}]+)}', r'$e^{-\\\1 \2}$', text)
        
        # Fix e^{-$\lambda$t} patterns
        text = re.sub(r'e\^\{-\$\\([a-zA-Z]+)\$([a-zA-Z]*)\}', r'$e^{-\\\1 \2}$', text)
        
        # Fix e^(-$\lambda$t) -> $e^{-\lambda t}$
        text = re.sub(r'e\^\(-\$\\([a-zA-Z]+)\$([a-zA-Z]*)\)', r'$e^{-\\\1 \2}$', text)
        
        # Fix spacing: add space between $...$ and immediately following letter
        # Pattern requires content to not start with whitespace (avoids matching across $ pairs)
        text = re.sub(r'\$([^\s$][^$]*)\$([a-zA-Z])', r'$\1$ \2', text)
        
        # Clean up empty math
        text = re.sub(r'\$\s*\$', '', text)
        
        # Clean up double dollars
        text = re.sub(r'\$\$', '$', text)
        
        # Fix double-wrapped
        text = re.sub(r'\$\$([^$]+)\$\$', r'$\1$', text)
        
        return text
    
    def convert_to_latex(text):
        """Apply all conversions."""
        text = convert_unicode_to_latex(text)
        text = convert_html_to_latex(text)
        text = clean_markdown(text)
        text = wrap_latex_in_dollars(text)
        text = merge_adjacent_math(text)
        text = post_process(text)
        return text
    
    def process_obj(obj):
        """Recursively process JSON object."""
        if isinstance(obj, dict):
            return {k: process_obj(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_obj(item) for item in obj]
        elif isinstance(obj, str):
            return convert_to_latex(obj)
        return obj
    
    # Process file
    line_num = 0
    with open(input_file, 'r', encoding='utf-8') as f_in:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            for line_num, line in enumerate(f_in, 1):
                try:
                    obj = json.loads(line)
                    obj = process_obj(obj)
                    json.dump(obj, f_out, ensure_ascii=False)
                    f_out.write('\n')
                except Exception as e:
                    print(f"Error on line {line_num}: {e}")
                    f_out.write(line)
    
    print(f"Processed {line_num} lines -> {output_file}")
    return output_file

In [3]:
fix_latex('../content/questions_dataset.jsonl', '../content/questions_dataset_cleaned.jsonl')

Processed 66 lines -> ../content/questions_dataset_cleaned.jsonl


'../content/questions_dataset_cleaned.jsonl'